# Preprocessing

The text content is not in its correct format - it is currently in a word format. This data cannot be trained on, so we must put it through a process called tokenisation to convert it into a format the model will understand (this will be explained throughout the notebooks). 

Of course, tokenisation is not possible without first cleaning up the data. 

## Imports

Firstly, let's import our data as a CSV. 

In [23]:
import pandas as pd
df: pd.DataFrame = pd.read_csv("csv/all_data.csv")

In this section, I am using a library called NLTK (Natural Language Toolkit), which contains utilities for being able to convert sentences into tokens.   

In [24]:
import nltk
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
import re
import contractions

This downloads samples and datasets for the nltk to use. I'm not particularly sure about what it does, NLTK functions throw an error asking for this to be downloaded. 

In [25]:
nltk.download('stopwords')
nltk.download('twitter_samples')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package twitter_samples to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Stopwords

Stopwords are words that carry semantic meaning and can make analysis of the words hard. It includes such articles ("a", "the", ...), conjunctions ("and", "or", ...), prepositions ("in", "on", ...) and other such words. 

By removing stopwords, we are able to remove any tokens/words that do not hold minimal meaning. 

In [26]:
stop_words = set(stopwords.words('english'))

There are some words that can drastically change the meaning if removed. 

For example:

"That is not good" (negative) ----removing stopword---> "That is good" (positive) *[not what we want]*

By excluding the negations from the stopwords, we will be able to keep the sentiment of the text block

In [27]:
negations = {'no', 'not', 'nor', 'neither', 'never', 'none',
            "don't", "won't", "can't", "isn't", "aren't", "wasn't"}
stop_words = stop_words - negations

## Tokenising

By tokenising, we are able to convert a long block of text into individual words. 

In [28]:
tokeniser = TweetTokenizer(strip_handles=True, reduce_len=True)

In [29]:
def clean_text_content(text):
    if pd.isna(text):
        return ""
    text = str(text)

    text = re.sub(r'[^\x00-\x7F]+', '', text)     # remove non-ASCII
    text = re.sub(r'http\S+|www\.\S+', '', text)  # remove URLs
    text = re.sub(r'#\w+', '', text)              # remove hashtags
    text = re.sub(r'&\w+;', '', text)             # remove HTML entities
    text = contractions.fix(text)                 # expand contractions
    text = text.lower().strip()
    return text

df['Mutated Text Content'] = df['Text Content'].apply(clean_text_content)
df.head()

,Unnamed: 0,Sentiment,Text Content,tweet_length,Mutated Text Content
0,0,Positive,Grinding.. twitch.tv / krrnster,31,grinding.. twitch.tv / krrnster
1,1,Irrelevant,This was part of a 2006 @ colsonwhitehead nove...,71,this was part of a 2006 @ colsonwhitehead nove...
2,2,Irrelevant,I did this trick a couple of months ago hahaha...,418,i did this trick a couple of months ago hahaha...
3,3,Negative,Am I the only one who experienced so many mist...,69,am i the only one who experienced so many mist...
4,4,Neutral,Hearthstone’s Dragoncaster didn’t make a berth...,91,hearthstones dragoncaster did not make a berth...


In [30]:
def tokenise_text_content(text):
    if not text:
        return []
    tokens = tokeniser.tokenize(text)
    tokens = [re.sub(r'[^\w\s]', '', w) for w in tokens]  # remove punctuation
    tokens = [w for w in tokens if w and w not in stop_words]
    return tokens

df["Text Tokens"] = df["Mutated Text Content"].apply(tokenise_text_content)

# remove rows with no usable tokens like empty
df = df[df["Text Tokens"].apply(len) > 0].reset_index(drop=True)

df.head()

,Unnamed: 0,Sentiment,Text Content,tweet_length,Mutated Text Content,Text Tokens
0,0,Positive,Grinding.. twitch.tv / krrnster,31,grinding.. twitch.tv / krrnster,"[grinding, twitchtv, krrnster]"
1,1,Irrelevant,This was part of a 2006 @ colsonwhitehead nove...,71,this was part of a 2006 @ colsonwhitehead nove...,"[part, 2006, colsonwhitehead, novel, apex, hid..."
2,2,Irrelevant,I did this trick a couple of months ago hahaha...,418,i did this trick a couple of months ago hahaha...,"[trick, couple, months, ago, hahahahahahahahah..."
3,3,Negative,Am I the only one who experienced so many mist...,69,am i the only one who experienced so many mist...,"[one, experienced, many, mistakes, ghostrecon]"
4,4,Neutral,Hearthstone’s Dragoncaster didn’t make a berth...,91,hearthstones dragoncaster did not make a berth...,"[hearthstones, dragoncaster, not, make, berth,..."


## Classification of sentiment

As previously mentioned, there are ~~4~~ 3 primary types of sentiment classification: 
- Positive
- Neutral/Irrelevant
- Negative

Within the set, we can use multi-class classification as so:
- Positive (2)
- Negative (1) (it only made sense if neutral was 0, not Negative)
- Neutral/Irrelevant (0)

In [31]:
s_map = {
    'Positive': 2,
    'Negative': 1,
    'Neutral': 0,
    'Irrelevant': 0,
}

df['Sentiment'] = df['Sentiment'].map(s_map)
df.head()

,Unnamed: 0,Sentiment,Text Content,tweet_length,Mutated Text Content,Text Tokens
0,0,2,Grinding.. twitch.tv / krrnster,31,grinding.. twitch.tv / krrnster,"[grinding, twitchtv, krrnster]"
1,1,0,This was part of a 2006 @ colsonwhitehead nove...,71,this was part of a 2006 @ colsonwhitehead nove...,"[part, 2006, colsonwhitehead, novel, apex, hid..."
2,2,0,I did this trick a couple of months ago hahaha...,418,i did this trick a couple of months ago hahaha...,"[trick, couple, months, ago, hahahahahahahahah..."
3,3,1,Am I the only one who experienced so many mist...,69,am i the only one who experienced so many mist...,"[one, experienced, many, mistakes, ghostrecon]"
4,4,0,Hearthstone’s Dragoncaster didn’t make a berth...,91,hearthstones dragoncaster did not make a berth...,"[hearthstones, dragoncaster, not, make, berth,..."


# Finish

Preprocessing is completed. Let's save into a .csv to be modified in feature_engineering

In [32]:
df.to_csv('csv/preprocess.csv', index=False)